# GRAM — Paper-Scale Training (N-Queens 8×8)

Reproduction of *Generative Recursive Reasoning* (Baek et al., arXiv:2605.19376).

Files this notebook expects in the same directory:
- `gram_model.py` — model (RoPE + ACT halt + EMA + LPRM all on)
- `data_nqueens.py` — 92-solution batcher

Note: module is named `gram_model` (not `gram`) to avoid collision with the PyPI `gram` package.

Pipeline:
1. Paper config: D=512, h=8, K=4, T=3, N_sup=16, β=0.07, kl_balance=0.8
2. AdamW lr=1e-4, wd=1.0, EMA=0.9999
3. Train ~50k steps; eval every 2k with both best-of-1 and best-of-20 + halt
4. Save checkpoint with EMA shadow weights

Hardware: needs a single ~24GB GPU. A4000/4090/A100 all fine.

In [ ]:
import os, time, math, json, torch
from torch.optim import AdamW
from gram_model import GRAM, GRAMConfig, EMA
from data_nqueens import NQueensBatcher

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
if device == 'cuda':
    print('gpu  :', torch.cuda.get_device_name(0))
    print('vram :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
torch.manual_seed(0)

In [ ]:
# Paper-scale config
cfg = GRAMConfig(
    vocab_size = 3,
    seq_len    = 64,        # 8x8 board flattened
    d_model    = 512,
    n_heads    = 8,
    ffn_hidden = 512,
    n_layers   = 2,
    K          = 4,
    T          = 3,
    N_sup      = 16,
    use_attn   = True,
    use_rope   = True,
    use_halt   = True,
)
model = GRAM(cfg).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'params           : {n_params/1e6:.2f}M')
print(f'transitions/step : N_sup * T = {cfg.N_sup * cfg.T}')

ema = EMA(model, decay=0.9999)

# Paper hparams: lr=1e-4, wd=1.0
opt = AdamW(model.parameters(), lr=1e-4, weight_decay=1.0, betas=(0.9, 0.95))

batcher = NQueensBatcher(n=8)
print(f'n-queens solutions: {batcher.num_solutions()}')

In [ ]:
# Training hyperparameters
B           = 128       # batch size — bump up if VRAM allows
n_steps     = 50_000
warmup      = 1_000
log_every   = 200
eval_every  = 2_000
ckpt_every  = 5_000
beta        = 0.07
kl_balance  = 0.8
halt_weight = 0.5

def lr_at(step):
    """Linear warmup then constant."""
    if step < warmup:
        return step / max(1, warmup)
    return 1.0

@torch.no_grad()
def _accuracy(pred, y, x, mask_token):
    mask_pos = (x == mask_token)
    correct  = (pred == y)
    full_tok   = correct.float().mean().item()
    mask_tok   = (correct & mask_pos).float().sum().item() / max(mask_pos.sum().item(), 1)
    full_board = correct.all(dim=1).float().mean().item()
    return full_tok, mask_tok, full_board

@torch.no_grad()
def evaluate(model, batcher, batch_size, device, N_best=20, use_ema=False, ema=None):
    model.eval()
    x, y = batcher.sample(batch_size)
    x, y = x.to(device), y.to(device)

    ctx = ema.swap_in(model) if use_ema else _nullcontext()
    with ctx:
        logits1 = model(x)
        ft1, mt1, fb1 = _accuracy(logits1.argmax(-1), y, x, batcher.MASK)
        logitsN, scores = model.forward_best_of_n(x, N=N_best)
        ftN, mtN, fbN = _accuracy(logitsN.argmax(-1), y, x, batcher.MASK)
        if model.cfg.use_halt:
            logitsH, n_steps = model.forward_with_halt(x)
            ftH, mtH, fbH = _accuracy(logitsH.argmax(-1), y, x, batcher.MASK)
            avg_steps = n_steps.float().mean().item()
        else:
            ftH = mtH = fbH = avg_steps = float('nan')
    return dict(
        n1_full_token=ft1, n1_mask_token=mt1, n1_full_board=fb1,
        nN_full_token=ftN, nN_mask_token=mtN, nN_full_board=fbN,
        halt_full_token=ftH, halt_full_board=fbH,
        halt_avg_steps=avg_steps, score_mean=scores.mean().item(),
    )

from contextlib import contextmanager
@contextmanager
def _nullcontext():
    yield

In [ ]:
# Training loop
log = []
t0 = time.time()

for step in range(1, n_steps + 1):
    # warmup lr
    for g in opt.param_groups:
        g['lr'] = 1e-4 * lr_at(step)

    x, y = batcher.sample(B)
    x, y = x.to(device), y.to(device)

    model.train()
    loss, info = model.train_step(
        x, y, beta=beta, kl_balance=kl_balance, halt_weight=halt_weight)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    gn = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    ema.update(model)

    if step == 1 or step % log_every == 0:
        msg = (f'step {step:6d} | loss {info["loss"]:.4f} | recon {info["recon"]:.4f} | '
               f'kl {info["kl"]:.4f} | halt {info["halt"]:.4f} | lprm {info["lprm"]:.4f} | '
               f'r {info["r"]:.3f} | acc {info["acc"]:.3f} | gn {gn.item():.2f} | '
               f't {time.time()-t0:.1f}s')
        print(msg)
        log.append(dict(step=step, **info, gnorm=gn.item(), t=time.time()-t0))

    if step % eval_every == 0:
        ev_raw = evaluate(model, batcher, 128, device, N_best=20, use_ema=False)
        ev_ema = evaluate(model, batcher, 128, device, N_best=20, use_ema=True, ema=ema)
        print('  >> raw  N=1  full_tok %.3f mask %.3f board %.3f' % (
            ev_raw['n1_full_token'], ev_raw['n1_mask_token'], ev_raw['n1_full_board']))
        print('  >> raw  N=20 full_tok %.3f mask %.3f board %.3f' % (
            ev_raw['nN_full_token'], ev_raw['nN_mask_token'], ev_raw['nN_full_board']))
        print('  >> raw  halt full_tok %.3f board %.3f avg_steps %.2f' % (
            ev_raw['halt_full_token'], ev_raw['halt_full_board'], ev_raw['halt_avg_steps']))
        print('  >> EMA  N=1  full_tok %.3f mask %.3f board %.3f' % (
            ev_ema['n1_full_token'], ev_ema['n1_mask_token'], ev_ema['n1_full_board']))
        print('  >> EMA  N=20 full_tok %.3f mask %.3f board %.3f' % (
            ev_ema['nN_full_token'], ev_ema['nN_mask_token'], ev_ema['nN_full_board']))
        print('  >> EMA  halt full_tok %.3f board %.3f avg_steps %.2f' % (
            ev_ema['halt_full_token'], ev_ema['halt_full_board'], ev_ema['halt_avg_steps']))

    if step % ckpt_every == 0:
        torch.save({
            'cfg': cfg.__dict__,
            'model': model.state_dict(),
            'ema':   ema.state_dict(),
            'opt':   opt.state_dict(),
            'step':  step,
        }, f'gram_paper_step{step}.pt')
        print(f'  >> ckpt saved: gram_paper_step{step}.pt')

torch.save({
    'cfg': cfg.__dict__,
    'model': model.state_dict(),
    'ema':   ema.state_dict(),
    'log':   log,
}, 'gram_paper_final.pt')
print('done — saved gram_paper_final.pt')

## Final evaluation

Pass criteria for paper-scale (per the GRAM paper Table 2 and our reading):
- **EMA best-of-1 full_token > 0.99**
- **EMA best-of-20 full_board >> EMA best-of-1 full_board**  ← the LPRM payoff
- **Halt avg_steps < N_sup** when full_board is high — model has learned to stop

In [ ]:
ev = evaluate(model, batcher, 512, device, N_best=20, use_ema=True, ema=ema)
print('FINAL EMA eval (batch=512):')
for k, v in ev.items():
    print(f'  {k:20s}: {v:.4f}')

In [ ]:
# Sanity: per-example score↔reward correlation. If LPRM is doing its job
# this should be > 0.7. If high but best-of-N==best-of-1, the prior has
# collapsed (no diversity to choose between).
import numpy as np
model.eval()
with ema.swap_in(model):
    x, y = batcher.sample(64)
    x, y = x.to(device), y.to(device)
    e_x = model.encode(x).repeat_interleave(20, dim=0)
    h_T = model._run_prior_trajectory(e_x)
    scores = model.value_head(h_T).view(64, 20).cpu().numpy()
    logits = model.lm_head(model.norm_out(h_T))
    preds  = logits.argmax(-1).view(64, 20, -1)
    target = y[:, None, :].expand(-1, 20, -1)
    rewards = (preds == target).float().mean(-1).cpu().numpy()

    diversity = np.array([len(set(map(tuple, preds[i].tolist()))) for i in range(64)])

from numpy import corrcoef
print('mean distinct preds / 20 samples (>1 means prior has diversity):', diversity.mean())
print('corr(score, reward) flat                                       :',
      corrcoef(scores.flatten(), rewards.flatten())[0,1])